# Score por reglas sobre el dataset real — prueba rápida

**Notebook de prueba, sábado 19 sep (ML-2).** El pipeline vive en `xray.labels` y `xray.rules` (spec: `docs/rules_spec.md`); aquí solo se importa y se mira. Como `features.build()` es del slice #2 y aún no existe, la §1 construye una **tabla provisional** con las cuatro señales a partir de los CSV, reutilizando la lógica del notebook 01. Es desechable: cuando ML-1 entregue la tabla real, la §1 se borra y el resto sigue igual.

Diferencias conocidas con el contrato: `min_balance_eur` es el mínimo de saldo al cierre de los días con movimiento (no de todos los días); el stock de vencidas se reconstruye con `due_date` y `payment_date` de las pagadas, como en el notebook 01 §11.

**19 sep (mañana), tras la revisión del score:** la §1 corta en el último mes completo (2026-08; 2026-09 solo tiene el día 1) y añade las tres señales v2 con `features.overdue_flow_rate()` y `features.derive()`. El resto del notebook sigue igual: `rules.run` lee las nuevas columnas.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from xray import features, labels, rules
from xray.data import load, artifacts_dir

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
FULL_MONTHS = pd.period_range("2024-09", "2026-09", freq="M")
OPERATING_IN = ["collection", "bulk_collection", "pos_settlement", "cash_settlement"]

## 1. Tabla provisional `features(company_id, month)` — desechable hasta el slice #2

In [ ]:
t = load()
companies, bank, debt, tx, inv, bal = (t[k] for k in ["companies", "banking_products", "debt_products", "transactions", "invoices", "balances"])
tx = tx[tx["date"].notna()].copy()
tx["month"] = tx["date"].dt.to_period("M")
print(f"{len(tx):,} movimientos · {tx['company_id'].nunique()} empresas con movimientos")

In [ ]:
# --- grano: meses desde el primero hasta el último con movimientos, sin huecos --------------
act = tx.groupby(["company_id", "month"]).size().unstack().reindex(columns=FULL_MONTHS)
first = act.notna().idxmax(axis=1)
last = act.notna().iloc[:, ::-1].idxmax(axis=1)
grid = pd.DataFrame({"company_id": np.repeat(first.index, len(FULL_MONTHS)),
                     "month": np.tile(FULL_MONTHS, len(first))})
grid = grid[(grid["month"] >= grid["company_id"].map(first)) & (grid["month"] <= grid["company_id"].map(last))].reset_index(drop=True)
grid["months_of_history"] = grid.groupby("company_id").cumcount() + 1
print(f"{len(grid):,} filas empresa-mes · mediana {grid.groupby('company_id').size().median():.0f} meses")

In [ ]:
# --- flujos del mes -----------------------------------------------------------------------
pos = tx[(tx["amount"] > 0) & tx["category"].isin(OPERATING_IN)]
flows = pd.DataFrame({
    "operating_inflows_eur": pos.groupby(["company_id", "month"])["amount"].sum(),
    "outflows_eur": -tx[tx["amount"] < 0].groupby(["company_id", "month"])["amount"].sum(),
})
feat = grid.merge(flows, left_on=["company_id", "month"], right_index=True, how="left").fillna({"operating_inflows_eur": 0.0, "outflows_eur": 0.0})

# --- saldo reconstruido de las cuentas corrientes: cierre de mes y mínimo del mes -------------
chk = bank.loc[bank["type"] == "checking", "product_id"]
chk_bal = bal[bal["product_id"].isin(chk) & bal["balance"].notna()].drop_duplicates("product_id")
final_by_company = chk_bal.groupby("company_id")["balance"].sum()
ctx = tx[tx["product_id"].isin(chk_bal["product_id"])]
daily = ctx.groupby(["company_id", "date"])["amount"].sum().sort_index()
# saldo al cierre del día d = saldo final − Σ movimientos posteriores a d
after = daily.groupby(level=0).transform(lambda s: s[::-1].cumsum()[::-1] - s)
eod = (final_by_company.reindex(daily.index.get_level_values(0)).values - after).rename("eod")
eod = eod.reset_index()
eod["month"] = eod["date"].dt.to_period("M")
m = eod.groupby(["company_id", "month"])["eod"].agg(eom_balance_eur="last", min_in_month="min")
m = m.reindex(pd.MultiIndex.from_frame(grid[["company_id", "month"]]))
m["eom_balance_eur"] = m.groupby(level=0)["eom_balance_eur"].ffill()  # meses sin movimiento: saldo arrastrado
carry = m.groupby(level=0)["eom_balance_eur"].shift(1)
m["min_balance_eur"] = np.fmin(m["min_in_month"].to_numpy(), carry.to_numpy())
m["min_balance_eur"] = m["min_balance_eur"].fillna(m["eom_balance_eur"])
feat = feat.merge(m[["eom_balance_eur", "min_balance_eur"]], left_on=["company_id", "month"], right_index=True, how="left")
neg = feat["min_balance_eur"].lt(0).astype(int)
feat["months_negative_6m"] = neg.groupby(feat["company_id"]).transform(lambda s: s.rolling(6, min_periods=1).sum()).astype(int)
print(f"saldo reconstruido para {feat['eom_balance_eur'].notna().groupby(feat['company_id']).any().sum()} empresas con cuenta corriente en balances")

In [ ]:
# --- vencidas a proveedores (notebook 01 §11) --------------------------------------------------
rec = inv[(inv["direction"] == "received") & inv["due_date"].notna()].copy()
rec["a"] = rec["amount"].abs()
rec["due_m"] = rec["due_date"].dt.to_period("M")
paid_ok = rec["status"].eq("paid") & rec["payment_date"].notna()
rec["rel_m"] = pd.concat([rec["due_m"], rec["payment_date"].dt.to_period("M")], axis=1).max(axis=1).where(paid_ok)
rec["iss_m"] = rec["issuance_date"].dt.to_period("M")

def cum_by_month(frame, col):
    ev = frame.dropna(subset=[col]).groupby(["company_id", col])["a"].sum().unstack()
    rng = pd.period_range(min(ev.columns.min(), FULL_MONTHS[0]), max(ev.columns.max(), FULL_MONTHS[-1]), freq="M")
    return ev.reindex(columns=rng).fillna(0).cumsum(axis=1)[FULL_MONTHS]

due_c, rel_c = cum_by_month(rec, "due_m"), cum_by_month(rec, "rel_m")
overdue_stock = (due_c - rel_c.reindex(due_c.index).fillna(0)).clip(lower=0)
received_3m = (rec.groupby(["company_id", "iss_m"])["a"].sum().unstack().reindex(columns=FULL_MONTHS).fillna(0)
                  .T.rolling(3, min_periods=1).sum().T).reindex(overdue_stock.index)
ov = pd.DataFrame({"overdue_received_eur": overdue_stock.stack(), "received_3m_eur": received_3m.stack()})
ov.index = ov.index.set_names(["company_id", "month"])
feat = feat.merge(ov, left_on=["company_id", "month"], right_index=True, how="left")
feat["has_invoices"] = feat["company_id"].isin(inv["company_id"].unique())
feat["overdue_received_ratio_3m"] = (feat["overdue_received_eur"] / feat["received_3m_eur"].where(feat["received_3m_eur"] > 0))

# --- servicio de deuda y DSCR (notebook 01 §12) -------------------------------------------------
svc = (tx[tx["category"].isin(["debt_repayment", "interest_charge"])].assign(a=lambda d: -d["amount"])
         .groupby(["company_id", "month"])["a"].sum())
feat["service_m"] = feat.set_index(["company_id", "month"]).index.map(svc).to_numpy()
feat["service_m"] = feat["service_m"].fillna(0.0)
g = feat.groupby("company_id")
feat["debt_service_6m_eur"] = g["service_m"].transform(lambda s: s.rolling(6, min_periods=1).sum())
infl_6m = g["operating_inflows_eur"].transform(lambda s: s.rolling(6, min_periods=1).sum())
feat["has_debt"] = g["service_m"].cumsum().gt(0)
feat["dscr_6m"] = (infl_6m / feat["debt_service_6m_eur"].where(feat["debt_service_6m_eur"] > 0))

# --- entradas interanuales ----------------------------------------------------------------------
prev = feat[["company_id", "month", "operating_inflows_eur"]].copy()
prev["month"] = prev["month"] + 12
feat = feat.merge(prev.rename(columns={"operating_inflows_eur": "inflows_prev_year"}), on=["company_id", "month"], how="left")
feat["has_prior_year"] = feat["inflows_prev_year"].notna()
feat["inflows_yoy_change"] = feat["operating_inflows_eur"] / feat["inflows_prev_year"].where(feat["inflows_prev_year"] > 0) - 1

In [ ]:
# --- extras: uso de línea (notebook 01 §10) y cliente principal (§13) ------------------------------
loc_b = (debt.loc[debt["type"] == "lineofcredit", ["product_id", "company_id", "granted"]].dropna().query("granted != 0")
             .merge(bal[["product_id", "balance"]].dropna(), on="product_id").drop_duplicates("product_id"))
d = tx[tx["product_id"].isin(loc_b["product_id"])]
lm = d.groupby(["product_id", "month"])["amount"].sum().unstack().reindex(index=loc_b["product_id"], columns=FULL_MONTHS).fillna(0)
after_l = lm.iloc[:, ::-1].cumsum(axis=1).iloc[:, ::-1] - lm
drawn = pd.DataFrame(-(loc_b["balance"].values[:, None] - after_l.values), index=loc_b["product_id"], columns=FULL_MONTHS)
drawn = drawn.where(lm.ne(0).cummax(axis=1))[lm.ne(0).any(axis=1)]
gr = loc_b.set_index("product_id").loc[drawn.index]
gmat = pd.DataFrame(np.where(drawn.notna(), gr["granted"].abs().values[:, None], np.nan), index=drawn.index, columns=FULL_MONTHS)
usage = (drawn.groupby(gr["company_id"].values).sum(min_count=1) / gmat.groupby(gr["company_id"].values).sum(min_count=1)).clip(0, 1)
usage = usage.stack().rename("credit_line_usage"); usage.index = usage.index.set_names(["company_id", "month"])
feat = feat.merge(usage, left_on=["company_id", "month"], right_index=True, how="left")
feat["has_credit_line"] = feat["company_id"].isin(debt.loc[debt["type"] == "lineofcredit", "company_id"].unique())

iss = inv[(inv["direction"] == "issued") & inv["counterparty_id"].notna() & inv["issuance_date"].notna()]
iss = iss.assign(month=iss["issuance_date"].dt.to_period("M"), a=iss["amount"].abs())
pc = iss.groupby(["company_id", "counterparty_id", "month"])["a"].sum().unstack().reindex(columns=FULL_MONTHS).fillna(0)
r12 = pc.T.rolling(12, min_periods=6).sum().T
tot12 = r12.groupby(level="company_id").transform("sum")
top1 = (r12 / tot12.where(tot12 > 0)).groupby(level="company_id").max().stack().rename("top_customer_share_12m")
top1.index = top1.index.set_names(["company_id", "month"])
feat = feat.merge(top1, left_on=["company_id", "month"], right_index=True, how="left")

In [ ]:
# --- al contrato: NaN donde no hay cobertura, tipos, orden de columnas, validate() -------------
feat["month"] = feat["month"].astype(str)
for col in ("overdue_received_eur", "received_3m_eur", "overdue_received_ratio_3m", "top_customer_share_12m"):
    feat.loc[~feat["has_invoices"], col] = np.nan
for col in ("debt_service_6m_eur", "dscr_6m"):
    feat.loc[~feat["has_debt"], col] = np.nan
feat.loc[~feat["has_credit_line"], "credit_line_usage"] = np.nan
feat.loc[~feat["has_prior_year"], "inflows_yoy_change"] = np.nan
feat["top_customer_share_12m"] = feat["top_customer_share_12m"].clip(upper=1)
feat["inflows_yoy_change"] = feat["inflows_yoy_change"].clip(lower=-1)
# empresas sin cuenta corriente en balances: sin saldo reconstruible → fuera de la tabla provisional
feat = feat[feat["min_balance_eur"].notna()].copy()
feat["months_of_history"] = feat.groupby("company_id").cumcount() + 1
# --- señales v2 (19 sep): último mes completo, tasa de vencidas desde facturas, derivadas del contrato
feat = feat[feat["month"] <= "2026-08"].copy()  # 2026-09 solo tiene el día 1: no es un mes
feat = feat.merge(features.overdue_flow_rate(inv, FULL_MONTHS), on=["company_id", "month"], how="left")
feat.loc[~feat["has_invoices"], "overdue_flow_rate_3m"] = np.nan
feat = features.derive(feat)
feat = feat[features.COLUMN_NAMES].reset_index(drop=True)
features.validate(feat)
feat.to_parquet(artifacts_dir() / "features_quick.parquet", index=False)
print(f"OK contrato · {len(feat):,} filas · {feat['company_id'].nunique()} empresas")
feat.describe().T.round(2)

## 2. Pipeline de reglas sobre la tabla provisional

`rules.run` ranquea dentro del mes, calcula el índice, el evento y la etiqueta, ajusta el mapa isotónico con `month ≤ 2025-08` y devuelve la tabla plana.

In [ ]:
cfg = rules.RulesConfig()
out = rules.run(feat, cfg=cfg, train_until="2025-08")
model = rules.fit(out, cfg, train_until="2025-08")
print(f"mapa isotónico: {len(model.knots_x)} nudos sobre {model.n_train:,} filas de train · lead_cutoff = {model.lead_cutoff:.1f}")
print(f"eventos: {out['event'].sum()} en {out.loc[out['event'], 'company_id'].nunique()} empresas (preview del notebook 01: 222 en 177 con umbrales absolutos)")
print(f"filas con score: {out['score'].notna().mean():.0%} · con etiqueta t+6: {out['label_t6'].notna().mean():.0%}")
pd.DataFrame({"outlook": out["outlook"].value_counts(normalize=True).round(3),
              "confidence": out["confidence"].value_counts(normalize=True).round(3)})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
grid_x = np.linspace(0, 1, 200)
axes[0].plot(grid_x, model.predict(grid_x)); axes[0].set_title("mapa isotónico nivel → score"); axes[0].set_xlabel("nivel_t"); axes[0].set_ylabel("score")
sc = out.dropna(subset=["score"])
q = sc.groupby("month")["score"].quantile([.1, .25, .5, .75, .9]).unstack()
axes[1].plot(q.index.astype(str), q[.5], color="k"); axes[1].fill_between(q.index.astype(str), q[.25], q[.75], alpha=.3); axes[1].fill_between(q.index.astype(str), q[.1], q[.9], alpha=.15)
axes[1].set_title("score por mes: p10–p90, p25–p75, mediana"); axes[1].tick_params(axis="x", rotation=90)
red_share = (out["n_red"] >= cfg.red_month_min).groupby(out["month"]).mean()
axes[2].plot(red_share.index, red_share.values); axes[2].set_title("% empresas en mes rojo (≥ 2 señales)"); axes[2].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()

## 3. Evals rápidas (versión de andar por casa; la definitiva va a `xray/evals.py`)

In [ ]:
TEST = [str(p) for p in pd.period_range("2025-09", "2026-02", freq="M")]
o = out.sort_values(["company_id", "month"]).copy()
g = o.groupby("company_id")["event"]
aucs = {}
for h in range(1, 13):
    fut = sum(g.shift(-k).fillna(False).astype(bool) for k in range(1, h + 1)).astype(bool)
    complete = g.shift(-h).notna()
    mask = o["month"].isin(TEST) & ~o["in_event"] & o["score"].notna() & complete
    y, s = fut[mask], -o.loc[mask, "score"]
    aucs[h] = roc_auc_score(y, s) if y.nunique() == 2 else np.nan
aucs = pd.Series(aucs, name="AUC(h) test")
print(aucs.round(3).to_string())

In [ ]:
red = (o["n_red"] >= cfg.red_month_min)
gr = red.groupby(o["company_id"])
base = red.mean()
pers = pd.Series({k: gr.shift(-k)[red].astype(float).mean() for k in range(1, 13)}, name="P(rojo t+k | rojo t)")
lift = (pers / base).rename("lift")
horizon = int(lift[lift >= 2].index.max()) if (lift >= 2).any() else 0
print(f"tasa base de mes rojo {base:.1%} · horizonte de persistencia (lift ≥ 2×): {horizon} meses")
pd.concat([pers.round(3), lift.round(2)], axis=1).T

In [ ]:
# direccionalidad: Δscore(t−3→t) frente a Δnivel(t→t+6), en test
gs = o.groupby("company_id")
d_score = o["score"] - gs["score"].shift(3)
d_level = gs["level"].shift(-6) - o["level"]
m = o["month"].isin(TEST) & d_score.notna() & d_level.notna()
print(f"Spearman Δscore(t−3→t) vs Δnivel(t→t+6): {d_score[m].corr(d_level[m], method='spearman'):.3f} sobre {m.sum():,} filas")
neg = o["outlook"].eq("negative") & m
stab = o["outlook"].eq("stable") & m
print(f"P(nivel baja a 6 m | outlook negativo) = {(d_level[neg] < 0).mean():.0%} · P(baja | estable) = {(d_level[stab] < 0).mean():.0%}")

## 4. Dos empresas: una con evento, una sin

In [ ]:
with_event = out.loc[out["event"] & out["month"].isin(TEST), "company_id"].value_counts().index[:1].tolist()
quiet = out.groupby("company_id").filter(lambda d: len(d) >= 18 and d["n_red"].max() == 0)["company_id"].unique()[:1].tolist()
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, cid in zip(axes, with_event + quiet):
    d = out[out["company_id"] == cid].sort_values("month")
    ax.plot(d["month"], d["score"], marker="o", label="score")
    ax.plot(d["month"], d["level"] * 100, ls="--", label="nivel × 100")
    for mth in d.loc[d["event"], "month"]:
        ax.axvline(mth, color="red", alpha=.5)
    ax.set_title(f"{cid} · outlook final: {d['outlook'].iloc[-1]} · confidence: {d['confidence'].iloc[-1]}")
    ax.tick_params(axis="x", rotation=90); ax.legend()
plt.tight_layout(); plt.show()
out[out["company_id"].isin(with_event)][["month", "n_red", "state_index", "level", "score", "outlook", "event", "confidence"]].tail(10)

## 5. Lo mismo sobre la fixture de tres empresas (lo que ven los tests)

In [ ]:
fx = rules.run(features.load_fixture(), events_ext=pd.read_csv(features.FIXTURE_PATH.parent / "events_mock.csv"))
fx.pivot(index="month", columns="company_id", values="score").round(1).join(
    fx.pivot(index="month", columns="company_id", values="outlook").add_prefix("outlook_"))